In [ ]:
# ---------- Model 2: Heston FX ----------
def heston_call(S, K, T, rd, rf, v0, kappa, theta, xi, rho):
    """Heston FX call. price = S*e^(-rf*T)*P1 - K*e^(-rd*T)*P2, where P1 and P2 are
    'probabilities' obtained from the characteristic function and one numerical integral."""
    phi = np.linspace(1e-6, 12/np.sqrt(v0*T), 20000)               
    def prob(j):
        u, b = (0.5, kappa - rho*xi) if j == 1 else (-0.5, kappa)
        s = b - rho*xi*1j*phi
        d = np.sqrt(s**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (s - d)/(s + d)
        e = np.exp(-d*T)
        C = (rd - rf)*1j*phi*T + kappa*theta/xi**2*((s - d)*T - 2*np.log((1 - g*e)/(1 - g)))
        D = (s - d)/xi**2*(1 - e)/(1 - g*e)
        f = np.exp(C + D*v0 + 1j*phi*np.log(S))                    
        return 0.5 + trapezoid(np.real(np.exp(-1j*phi*np.log(K))*f/(1j*phi)), phi)/np.pi
    return S*np.exp(-rf*T)*prob(1) - K*np.exp(-rd*T)*prob(2)